In [1]:
%load_ext autoreload
%reload_ext autoreload

%autoreload 2
%matplotlib inline
import tensorflow
from tensorflow.compat.v1.keras.backend import get_session
tensorflow.compat.v1.disable_v2_behavior()
import math
import kerasAC 
from scipy.special import softmax,expit
from kerasAC.interpret.deepshap import * 
from kerasAC.interpret.profile_shap import * 
from kerasAC.vis import * 
from kerasAC.helpers.transform_bpnet_io import * 
from kerasAC.util import * 
import pandas as pd

## for plotting 

import matplotlib 
from matplotlib import pyplot as plt
plt.rcParams["figure.figsize"]=10,5
plt.rcParams['axes.xmargin'] = 0

font = {'family' : 'normal',
        'weight' : 'bold',
        'size'   : 10}

matplotlib.rc('font', **font)

Instructions for updating:
non-resource variables are not supported in the long term


Using TensorFlow backend.
/users/annashch/kerasAC/kerasAC/vis/plot_letters.py:172: FutureWarning: arrays to stack must be passed as a "sequence" type such as list or tuple. Support for non-sequence iterables such as generators is deprecated as of NumPy 1.16 and will raise an error in the future.
  min_coords = np.vstack(data.min(0) for data in polygons_data).min(0)
/users/annashch/kerasAC/kerasAC/vis/plot_letters.py:173: FutureWarning: arrays to stack must be passed as a "sequence" type such as list or tuple. Support for non-sequence iterables such as generators is deprecated as of NumPy 1.16 and will raise an error in the future.
  max_coords = np.vstack(data.max(0) for data in polygons_data).max(0)


In [2]:
#load the model! 
from keras.models import load_model
from keras.utils.generic_utils import get_custom_objects
from kerasAC.metrics import * 
from kerasAC.custom_losses import * 
custom_objects={"recall":recall,
                    "sensitivity":recall,
                    "specificity":specificity,
                    "fpr":fpr,
                    "fnr":fnr,
                    "precision":precision,
                    "f1":f1,
                    "ambig_binary_crossentropy":ambig_binary_crossentropy,
                    "ambig_mean_absolute_error":ambig_mean_absolute_error,
                    "ambig_mean_squared_error":ambig_mean_squared_error,
                    "MultichannelMultinomialNLL":MultichannelMultinomialNLL}
get_custom_objects().update(custom_objects)
model=load_model("/srv/scratch/annashch/chrombpnet/gm12878_dnase/with_bias_unplugged/gm12878.dnase.with.bias.unplugged.0.hdf5")

Instructions for updating:
If using Keras pass *_constraint arguments to layers.


In [3]:
#create the explainers 
model_wrapper=(model.input, model.outputs[1])
count_explainer=shap.DeepExplainer(model_wrapper,
                                   data=create_background_atac,
                                   combine_mult_and_diffref=combine_mult_and_diffref_1d)


In [4]:
prof_output = model.output[0]                                                                                                      
logits = prof_output - tf.reduce_mean(prof_output, axis=1, keepdims=True)
logits_stopgrad=tf.stop_gradient(logits)
probs=tf.nn.softmax(logits_stopgrad,axis=1)
logits_weighted=logits*probs 
prof_sum=tf.reduce_sum(logits_weighted,axis=(1,2))
prof_explainer=shap.DeepExplainer(model=(model.input,prof_sum),
                                  data=create_background_atac,
                                  combine_mult_and_diffref=combine_mult_and_diffref_1d)


In [17]:
from scipy.special import softmax 
def make_plot(label_counts_track,
              ref_pred_count_track,
              alt_pred_count_track,
              ref_pred_count_val,
              alt_pred_count_val,
              log_odds_prob,
              deepshap_prof_ref,
              deepshap_count_ref,
              deepshap_prof_alt,
              deepshap_count_alt,
              ism_prof_single_bp,
              ism_prof_zero_region,
              ism_prof_scrambled_region,
              ism_count_single_bp,
              ism_count_zero_region,
              ism_count_scrambled_region,
              ism_mat_observed,
              ism_mat_max,
              ism_mat_min,
              ism_mat_zero_region,
              ism_mat_scrambled_region,
              title,
              xmin=0,
              xmax=1346): 
    plt.rcParams["figure.figsize"]=12,25
    f,axes=plt.subplots(18, 1,sharex='row')
    plt.set_cmap('RdBu')
    
    #labels count track 
    axes[0].plot(label_counts_track)  
    axes[0].set_title("Label Counts")
    
    #alt & ref predicted count tracks 
    axes[1].plot(ref_pred_count_track,label='Ref pred counts per base',color='b')
    axes[1].plot(alt_pred_count_track,label='Alt pred counts per base',color='r')
    axes[1].set_title("Counts Pred Ref:"+str(ref_pred_count_val)+":"+"Counts Pred Alt:"+str(alt_pred_count_val))
    axes[1].legend() 
    
    #log-ratio predicted probs (blast radius track)
    axes[2].plot(log_odds_prob)
    axes[2].set_title("log(prediced prob(ref))-log(predicted prob(alt))")
    
    #plot DeepSHAP tracks for profile & counts, for ref & alt alleles 
    axes[3]=plot_bases_on_ax(deepshap_prof_ref,axes[3],show_ticks=False)
    axes[3].set_title('DeepSHAP Profile Ref')
    axes[3].set_ylim(-0.05,0.05)
    
    axes[4]=plot_bases_on_ax(deepshap_prof_alt,axes[4],show_ticks=False)
    axes[4].set_title('DeepSHAP Profile Alt')
    axes[4].set_ylim(-0.05,0.05)
    
    axes[5]=plot_bases_on_ax(deepshap_count_ref,axes[5],show_ticks=False)
    axes[5].set_title('DeepSHAP Count Ref')
    axes[5].set_ylim(-0.05,0.05)

    axes[6]=plot_bases_on_ax(deepshap_count_alt,axes[6],show_ticks=False)
    axes[6].set_title('DeepSHAP Count Alt')
    axes[6].set_ylim(-0.05,0.05)

    axes[7]=plot_bases_on_ax(ism_prof_single_bp,axes[7],show_ticks=False)
    axes[7].set_title("ISM Profile, single bp")
    
    axes[8]=plot_bases_on_ax(ism_prof_zero_region,axes[8],show_ticks=False)
    axes[8].set_title("ISM Profile, 5bp, zero region")
    
    axes[9]=plot_bases_on_ax(ism_prof_scrambled_region,axes[9],show_ticks=False)
    axes[9].set_title("ISM Profile, 5bp, scrambled region")
    
    axes[10]=plot_bases_on_ax(ism_count_single_bp,axes[10],show_ticks=False)
    axes[10].set_title("ISM Count, single bp")
    
    axes[11]=plot_bases_on_ax(ism_count_zero_region,axes[11],show_ticks=False)
    axes[11].set_title("ISM Count, 5bp, zero region")
    
    axes[12]=plot_bases_on_ax(ism_count_scrambled_region,axes[12],show_ticks=False)
    axes[12].set_title("ISM Count, 5bp, scrambled region")
    for i in range(13): 
        axes[i].set_xlim(xmin,xmax)
        axes[i].set_xticks(list(range(xmin, xmax, 50)))  
        
    extent = [0, ism_mat_observed.shape[0], 0, ism_mat_observed.shape[1]]
    ymin=np.amin(ism_mat_max)
    ymax=np.amax(ism_mat_min)
    abs_highest=max([abs(ymin),abs(ymax)])
    hmap1=axes[13].imshow(ism_mat_observed.T,
                        extent=extent,
                        vmin=-1*abs_highest,
                        vmax=abs_highest, 
                        interpolation='nearest',aspect='auto')
    axes[13].set_title("ISM matrix, single bp, observed allele projected")
    hmap2=axes[14].imshow(ism_mat_max.T,
                        extent=extent,
                        vmin=-1*abs_highest,
                        vmax=abs_highest, 
                        interpolation='nearest',aspect='auto')
    axes[14].set_title("ISM matrix,single bp, Max allele projected")
    hmap3=axes[15].imshow(ism_mat_min.T,
                        extent=extent,
                        vmin=-1*abs_highest,
                        vmax=abs_highest, 
                        interpolation='nearest',aspect='auto')
    axes[15].set_title("ISM matrix, single bp, Min allele projected")
    hmap4=axes[16].imshow(ism_mat_zero_region.T,
                        extent=extent,
                        vmin=-1*abs_highest,
                        vmax=abs_highest, 
                        interpolation='nearest',aspect='auto')
    axes[16].set_title("ISM matrix, 5bp zero")
    
    hmap5=axes[17].imshow(ism_mat_scrambled_region.T,
                        extent=extent,
                        vmin=-1*abs_highest,
                        vmax=abs_highest, 
                        interpolation='nearest',aspect='auto')
    axes[17].set_title("ISM matrix, 5bp scrambled")
    
    
    plt.suptitle(title)
    plt.tight_layout()
    plt.colorbar(hmap1,ax=axes[13],orientation='horizontal')
    plt.colorbar(hmap2,ax=axes[14],orientation='horizontal')
    plt.colorbar(hmap3,ax=axes[15],orientation='horizontal')
    plt.colorbar(hmap4,ax=axes[16],orientation='horizontal')
    plt.colorbar(hmap5,ax=axes[17],orientation='horizontal')
    plt.savefig(title.replace('/',':')+'.png',format='png',dpi=300)
    #plt.show()

In [6]:
import pysam 
ref=pysam.FastaFile("/mnt/data/GRCh38_no_alt_analysis_set_GCA_000001405.15.fasta")

In [7]:
import pyBigWig 


In [8]:
def get_deepshap(prof_explainer,count_explainer,seq_onehot): 
    profile_explanations=prof_explainer.shap_values(seq_onehot)*seq_onehot
    count_explanations=np.squeeze(count_explainer.shap_values(seq_onehot)[0])*seq_onehot  
    return profile_explanations, count_explanations

In [9]:
def get_preds(model,seq_onehot):
    preds=model.predict(seq_onehot)
    prof=np.squeeze(preds[0])
    probs=expit(prof)
    count=np.squeeze(preds[1])  
    count_track=probs*count 
    return prof,count,probs,count_track

In [10]:
import random
def get_ism_scrambled_region(model,seq,prof_pred,count_pred,mask_size=5): 
    #expand default preds to match dimensions for ISM 
    
    scrambled_seq=''.join(random.sample(seq,len(seq)))
    default_prof_expanded=np.zeros((len(seq),1000))
    default_count_expanded=np.zeros((len(seq)))
    for i in range(len(seq)): 
        default_prof_expanded[i,:]=prof_pred 
        default_count_expanded[i]=count_pred
            
    #create placeholders for ISM predictions        
    placeholder_prof=np.zeros((len(seq),1000))
    placeholder_count=np.zeros((len(seq)))

    for i in range(len(seq)): 
        mask_start=max([0,i-int(mask_size)//2])
        mask_end=min([len(seq),i+1+int(mask_size)//2])
        mask_length=mask_end-mask_start
        #print("mask length:"+str(mask_length))
        #print("mask start:"+str(mask_start))
        #print("mask end:"+str(mask_end))
        cur_seq=seq[0:mask_start]+scrambled_seq[mask_start:mask_end]+seq[mask_end::]
        #get predictions for the sequence with 0 mask  
        preds=model.predict([one_hot_encode([cur_seq])])
        prof=np.squeeze(preds[0])
        count=np.squeeze(preds[1])
        placeholder_prof[i,:]=prof
        placeholder_count[i]=count
    
    #subtract the WT, average across base axis 
    placeholder_prof_normed=-1*(placeholder_prof-default_prof_expanded)
    placeholder_count_normed=-1*(placeholder_count-default_count_expanded)
   
    seq_onehot=np.squeeze(one_hot_encode([seq]))
    ism_profile_track=np.expand_dims(np.sum(np.abs(placeholder_prof_normed),axis=1),axis=1)*seq_onehot
    ism_count_track=np.expand_dims(placeholder_count_normed,axis=1)*seq_onehot
    return ism_profile_track, ism_count_track, placeholder_prof_normed 

In [11]:
def get_ism_zero_region(model,seq,prof_pred,count_pred,mask_size=5): 
    #expand default preds to match dimensions for ISM 
    default_prof_expanded=np.zeros((len(seq),1000))
    default_count_expanded=np.zeros((len(seq)))
    for i in range(len(seq)): 
        default_prof_expanded[i,:]=prof_pred 
        default_count_expanded[i]=count_pred
            
    #create placeholders for ISM predictions        
    placeholder_prof=np.zeros((len(seq),1000))
    placeholder_count=np.zeros((len(seq)))

    for i in range(len(seq)): 
        mask_start=max([0,i-int(mask_size)//2])
        mask_end=min([len(seq),i+1+int(mask_size)//2])
        mask_length=mask_end-mask_start
        #print("mask length:"+str(mask_length))
        #print("mask start:"+str(mask_start))
        #print("mask end:"+str(mask_end))
        cur_seq=seq[0:mask_start]+"N"*mask_length+seq[mask_end::]
        #get predictions for the sequence with 0 mask  
        preds=model.predict([one_hot_encode([cur_seq])])
        prof=np.squeeze(preds[0])
        count=np.squeeze(preds[1])
        placeholder_prof[i,:]=prof
        placeholder_count[i]=count
    
    #subtract the WT, average across base axis 
    placeholder_prof_normed=-1*(placeholder_prof-default_prof_expanded)
    placeholder_count_normed=-1*(placeholder_count-default_count_expanded)

    seq_onehot=np.squeeze(one_hot_encode([seq]))
    ism_profile_track=np.expand_dims(np.sum(np.abs(placeholder_prof_normed),axis=1),axis=1)*seq_onehot
    ism_count_track=np.expand_dims(placeholder_count_normed,axis=1)*seq_onehot
    return ism_profile_track, ism_count_track, placeholder_prof_normed 

In [12]:
def get_ism_single_bp_sub(model,seq,prof_pred,count_pred): 
    #expand default preds to match dimensions for ISM 
    default_prof_expanded=np.zeros((len(seq),1000,4))
    default_count_expanded=np.zeros((len(seq),4))
    for j in range(4): 
        for i in range(len(seq)): 
            default_prof_expanded[i,:,j]=prof_pred 
            default_count_expanded[i,j]=count_pred
            
    #create placeholders for ISM predictions        
    ind_to_base={0:'A',1:'C',2:'G',3:'T'}
    placeholder_prof=np.zeros((len(seq),1000,4))
    placeholder_count=np.zeros((len(seq),4))

    for j in range(4):
        cur_allele_seqs=[]
        for i in range(len(seq)): 
            cur_allele_seqs.append(seq[0:i]+ind_to_base[j]+seq[i+1::])
        #get predictions for this allele 
        cur_allele_preds=model.predict([one_hot_encode(cur_allele_seqs)])
        cur_allele_prof=np.squeeze(cur_allele_preds[0])
        cur_allele_count=np.squeeze(cur_allele_preds[1])
        placeholder_prof[:,:,j]=cur_allele_prof
        placeholder_count[:,j]=cur_allele_count
    
    #subtract the WT, average across base axis 
    placeholder_prof_normed=placeholder_prof-default_prof_expanded
    placeholder_count_normed=placeholder_count-default_count_expanded

    placeholder_prof_normed=placeholder_prof_normed-np.expand_dims(np.mean(placeholder_prof_normed,axis=2),axis=2)
    placeholder_count_normed=placeholder_count_normed-np.expand_dims(np.mean(placeholder_count_normed,axis=1),axis=1)
    
    seq_onehot=one_hot_encode([seq])
    ism_profile_track=np.sum(np.abs(placeholder_prof_normed),axis=1)*seq_onehot
    ism_count_track=placeholder_count_normed*seq_onehot
    
    #ism heatmap: 
    #1) observed base 
    ism_mat_observed=np.sum(np.expand_dims(np.squeeze(seq_onehot),axis=1)*placeholder_prof_normed,axis=2)
    #2) max base 
    ism_mat_max=np.amax(placeholder_prof_normed,axis=2)
    #3) min base 
    ism_mat_min=np.amin(placeholder_prof_normed,axis=2)
    
    return ism_profile_track, ism_count_track, ism_mat_observed, ism_mat_max, ism_mat_min 

In [13]:
def analyze(ref,
            chrom,
            summit,
            ref_allele,
            alt_allele,
            rsid,
            bigwig,
            model,
            count_explainer,
            prof_explainer,
            flank=673):
    #get the reference and alternate one-hot-encoded sequences 
    seq=ref.fetch(chrom,summit-flank,summit+flank)
    
    ref_seq=seq[0:flank]+ref_allele+seq[flank+1::]
    assert len(ref_seq)==2*flank
    ref_onehot=one_hot_encode([ref_seq])
    
    alt_seq=seq[0:flank]+alt_allele+seq[flank+1::]
    assert len(alt_seq)==2*flank
    alt_onehot=one_hot_encode([alt_seq])
    
    #get the bigwig labels 
    labels=np.nan_to_num(bigwig.values(chrom,summit-flank,summit+flank))
    
    #get predictions for reference & alternate allele 
    prof_ref,count_ref,probs_ref,count_track_ref=get_preds(model,ref_onehot)
    prof_alt,count_alt,probs_alt,count_track_alt=get_preds(model,alt_onehot)
    
    #get the log odds blast radius track 
    blast_radius_track=[0]*173+(np.log(probs_ref)-np.log(probs_alt)).tolist()+[0]*173
    #get deepSHAP scores for ref & alt alleles 
    profile_explanations_ref, count_explanations_ref=get_deepshap(prof_explainer, count_explainer, ref_onehot)
    profile_explanations_alt, count_explanations_alt=get_deepshap(prof_explainer, count_explainer, alt_onehot)
    
    #get ISM scores 
    single_bp_ism_profile_track, single_bp_ism_count_track, single_bp_ism_mat_observed, single_bp_ism_mat_max, single_bp_ism_mat_min=get_ism_single_bp_sub(model,seq,prof_ref,count_ref)
    
    #get masked ISM scores (zero out the region)
    zero_region_ism_profile_track, zero_region_ism_count_track, zero_region_ism_mat =get_ism_zero_region(model,seq,prof_ref,count_ref)

    #get masked ISM scores (scrambled)
    scrambled_region_ism_profile_track, scrambled_region_ism_count_track, scrambled_region_ism_mat =get_ism_scrambled_region(model,seq,prof_ref,count_ref)
    
    #visualize 
    make_plot(labels,
              [0]*173+count_track_ref.tolist()+[0]*173,
              [0]*173+count_track_alt.tolist()+[0]*173,
              count_ref,
              count_alt,
              blast_radius_track,
              profile_explanations_ref,
              count_explanations_ref,
              profile_explanations_alt,
              count_explanations_alt,
              single_bp_ism_profile_track,
              zero_region_ism_profile_track,
              scrambled_region_ism_profile_track,
              single_bp_ism_count_track,
              zero_region_ism_count_track,
              scrambled_region_ism_count_track,
              single_bp_ism_mat_observed,
              single_bp_ism_mat_max,
              single_bp_ism_mat_min,
              zero_region_ism_mat,
              scrambled_region_ism_mat,
              ':'.join([str(chrom),str(summit),str(ref_allele),str(alt_allele),str(rsid)]))

In [14]:
#USE HG38 
regions=pd.read_csv("bQTL.for.ism.csv",header=0,sep='\t')
regions

,Chrom,hg19pos0,hg38pos0,Major,Minor,Group,Rsid
0,chr14,22590585,22122627,T,A,StrongbQTL,rs11157420
1,chr22,44116809,43720929,C,G,StrongbQTL,rs5764238
2,chr4,37814568,37812946,C,T,StrongbQTL,rs116009584
3,chr7,157313211,157520517,A,G,StrongbQTL,rs4716753
4,chr8,25242971,25385455,T,C,StrongbQTL,rs925032
5,chr1,64029605,63563934,G,A,StrongbQTL,rs217474
6,chr7,100230863,100633240,G,C,StrongbQTL,rs34242818
7,chr1,144534082,149162899,T,C,StrongbQTL,rs1184471061
8,chr6,26660654,26660426,G,C,StrongbQTL,rs2451713
9,chr15,52536308,52244111,G,A,Av,rs4774616


In [15]:
pbw=pyBigWig.open("/oak/stanford/groups/akundaje/projects/atlas/dnase_processed/dnase/13da5ebe-0941-4855-8599-40bbcc5c58b4/call-bowtie2/shard-0/execution/ENCSR000EMT.merged.bam.bpnet.unstranded.bw",'r')

In [18]:
import pdb
for index,row in regions.iterrows(): 
    chrom=row['Chrom']
    summit=row['hg38pos0']
    ref_allele=row['Major']
    alt_allele=row['Minor']
    rsid=row['Rsid']
    print(index)
    analyze(ref=ref,
            chrom=chrom,
            summit=summit,
            ref_allele=ref_allele,
            alt_allele=alt_allele,
            rsid=rsid,
            bigwig=pbw,
            model=model,
            count_explainer=count_explainer,
            prof_explainer=prof_explainer,
            flank=673)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
